# AI기반 소프트웨어 결함 예측모델

- 담당자: 김주영
- 작성일: 2026년 10월
- 데이터: NASA MDP (JM1 + CM1)
   
   https://www.kaggle.com/datasets/radowanulhaque/software-defect

# 1. 데이터 준비 단계

### ① 데이터 다운로드 및 병합

* JM1(약 10,885건) + CM1(약 498건)을 합쳐서 총 11,383건의 데이터셋 생성
* `project_source` 컬럼을 추가해 프로젝트명 구분
* 데이터 병합 이유
  1. CM1 단독으로는 샘플이 부족히므로, 학습 데이터 양 확보 필요.
  2. 특정 프로젝트에 과적합되는 것을 방지 및 성능 향상
  3. 실제 현장에서는 여러 프로젝트 데이터를 모아 학습하는 경우가 많으므로 실무 시나리오 반영

In [ ]:
# ==================== 1. 라이브러리 설치 및 데이터 로드 ====================
pip install kagglehub imbalanced-learn openpyxl -q

import kagglehub
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# 원천 데이터 다운로드
path = kagglehub.dataset_download("radowanulhaque/software-defect")

# 개별 프로젝트 파일 로드
df_jm1 = pd.read_csv(os.path.join(path, 'jm1.csv'))
df_cm1 = pd.read_csv(os.path.join(path, 'cm1.csv'))

# 데이터 출처 식별자 추가 및 병합
df_jm1['project_source'] = 'JM1'
df_cm1['project_source'] = 'CM1'
df_concat = pd.concat([df_jm1, df_cm1], ignore_index=True)

print(f"JM1 데이터: {len(df_jm1)}건")
print(f"CM1 데이터: {len(df_cm1)}건")
print(f"병합 완료. 총 데이터: {len(df_concat)}건")

In [ ]:
df_concat  # 병합된 데이터셋 검토

### ② 전처리

* `'?'`로 표시된 결측치를 제거하고, 최종 11,378건을 사용
* **`id` 컬럼 제거** (데이터 누수 방지)
    * id가 결함 여부와 강하게 연관되어 있어 제거하지 않으면 모델이 메트릭이 아닌 id만 보고 예측하게 됨
* `X` = 입력 피처 (코드 메트릭 21개)
* `y` = 타겟 (0: 정상, 1: 결함)

In [ ]:
# ==================== 2. 데이터 전처리 ====================
# 1. 결측치 정제 ('?' 기호를 NaN으로 변환 후 제거)
df_concat = df_concat.replace('?', np.nan)
df_concat = df_concat.dropna()

# 2. 피처(X)와 타겟(y) 분리
# 'id' 컬럼 제거 (데이터 누수 방지)
X = df_concat.drop(columns=['defects', 'project_source', 'id']).apply(pd.to_numeric)
y = df_concat['defects'].astype(int)  # True -> 1 (결함), False -> 0 (정상)

print(f"전처리 후 데이터 수: {len(X)}건")
print(f"사용 피처 수: {X.shape[1]}개")
print("\n클래스 분포:")
print(y.value_counts())

# 2. 시각화 (EDA)

1. **타겟 분포 확인** : 정상(0)과 결함(1)의 비율을 파악 → 불균형 여부를 확인하고 SMOTE 적용 여부를 결정
2. **피처 간 상관관계 확인** : 어떤 메트릭들이 서로 강하게 연결되어 있는지 파악 → 다중공선성 문제 사전 인지


    *   **현상:** 코드 지표들(길이, 부피, 복잡도 등)이 0.90~1.00 수준으로 심하게 중복됨
    *   **문제점:** 데이터 중복으로 인해 AI가 결함의 '진짜 원인(피처 중요도)'을 파악하지 못하고 결과가 왜곡
    *   **해결책:** 겹치는 지표 그룹에서 대표 지표 1개씩만 남기고 나머지는 모두 삭제해야 예측 성능과 분석력이 높아짐.





3. 데이터 분할 전에 수행하여 전체 데이터의 특성 파악


*   e (Effort): 유지보수 노력 = 정신적 피로도
*   n (Length): 프로그램 길이= 토큰 수
*   v (Volume): 프로그램 부피 = 코드의 물리적인 용량
*   b (Bugs): 예상 결함 수
    * ex) 코드의 부피(v)를 바탕으로 버그량을 통계적으로 추산한 값
*   total_Op (Total Operators): 총 연산자 수
    * ex) if, for, while, +, =, print()
*   total_Opnd (Total Operands): 총 피연산자 수
    * ex) x, y, count, "Hello" 등 개발자가 선언한 변수명이나 데이터 값의 총개수
*   branchCount: 분기점 수
    * ex) if-elif-else, for, try-except 등이 만들어내는 코드 로직의 갈래 개수  
*   lOCode (Lines of Code): 코드 줄 수





In [ ]:
# ==================== 3. 시각화 ====================
# 타겟 변수 불균형 분포 시각화
plt.figure(figsize=(6, 4))
sns.countplot(x=y, palette='Set2')
plt.title('Distribution of Target Variable (0: Normal, 1: Defect)')
plt.xlabel('Defect Status')
plt.ylabel('Count')
plt.show()

# 코드 복잡도 지표 간 상관관계 시각화 (Heatmap)
plt.figure(figsize=(14, 12))

corr_matrix = X.corr()

sns.heatmap(
    corr_matrix,
    annot=True,           # 상관계수 수치 표시
    fmt='.2f',            # 소수점 둘째 자리까지
    cmap='Blues',
    linewidths=0.5,
    annot_kws={"size": 8}
)

plt.title('Correlation Heatmap of Software Metrics', fontsize=14)
plt.tight_layout()
plt.show()

4. VIF(분산 팽창 지수) 기반 다중공선성 검증

*   VIF 10 이상 시 다중공선성 위험(Danger Threshold)으로 간주
*   uniq_Opnd와 v(g) 지표가 기준선을 근소하게 초과



In [ ]:
# [다중공선성 검증] Variance Inflation Factor (VIF) 분석
from statsmodels.stats.outliers_influence import variance_inflation_factor

print("\n[다중공선성 검증] Variance Inflation Factor (VIF) 분석")
vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]
vif_data = vif_data.sort_values(by="VIF", ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x="VIF", y="Feature", data=vif_data, palette="Reds_r")
plt.axvline(x=10, color='black', linestyle='--', linewidth=2, label='Danger Threshold (VIF=10)')
plt.title("Variance Inflation Factor (VIF) - Multicollinearity Check", fontsize=14)
plt.xlabel('VIF Score (Lower is better)')
plt.legend()
plt.tight_layout()
plt.show()

4. **다중 공선성 제거 (추가 전처리)**


*  상관계수 0.9 이상 및 VIF 무한대 수치를 유발하던 8개 중복 지표(e, n, v 등)삭제

*  남은 지표 간 최고 상관계수를 0.86 수준으로 낮추고, 대다수 지표를 VIF 안전 기준선(10 이하)으로 조정

*  데이터 중복에 따른 AI 모델의 과적합 방지 및 향후 출력될 피처 중요도의 분석 신뢰도 확보.


In [ ]:
# 상관관계 0.9 이상인 중복 지표(다중공선성 유발 피처) 제거
X = X.drop(columns=['e', 'n', 'v', 'b', 'total_Op', 'total_Opnd', 'branchCount', 'lOCode'])

In [ ]:
# ==================== 3-2. 전처리 후 VIF 안정화 재확인 ====================
from statsmodels.stats.outliers_influence import variance_inflation_factor

print("\n[다중공선성 검증] 전처리 후 Variance Inflation Factor (VIF) 재확인")
vif_data_reduced = pd.DataFrame()
vif_data_reduced["Feature"] = X.columns

# 정제된 X 데이터로 VIF 재계산
vif_data_reduced["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]
vif_data_reduced = vif_data_reduced.sort_values(by="VIF", ascending=False)

# VIF 막대그래프 시각화
plt.figure(figsize=(10, 8))
sns.barplot(x="VIF", y="Feature", data=vif_data_reduced, palette="Reds_r")

# 심각도 기준선(Threshold) 표시
plt.axvline(x=10, color='black', linestyle='--', linewidth=2, label='Danger Threshold (VIF=10)')
plt.title("VIF After Removing Multicollinearity", fontsize=14)
plt.xlabel('VIF Score (Lower is better)')
plt.legend()
plt.tight_layout()
plt.show()

# 3. 데이터 분할 및 불균형 처리 (SMOTE)

### ① Train / Test 분할
* 전체 데이터를 **8:2**로 분할
* `stratify=y` 옵션으로 학습/평가 세트의 결함 비율을 비슷하게 유지

### ② SMOTE 적용
* **학습 데이터에만** 적용 (평가 데이터는 원본 유지)
* 기존 학습 데이터: 정상 7,380 / 결함 1,722 (약 81%/ 약 19%)
* SMOTE 적용 후: 정상 7,380 / 결함 7,380 (1:1 균형)
* 평가 데이터에 SMOTE를 적용하지 않는 이유 → 실제 현장의 불균형 상황을 그대로 평가해야 하기 때문

In [ ]:
# ==================== 4. 데이터 분할 및 SMOTE ====================
# 학습용(Train)과 평가용(Test) 데이터 8:2 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("=== SMOTE 적용 전 학습 데이터 분포 ===")
print(y_train.value_counts())

# 훈련 데이터에만 SMOTE 적용 (평가용 데이터는 원본 유지)
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("\n=== SMOTE 적용 후 학습 데이터 분포 ===")
print(y_train_resampled.value_counts())

# 4. 모델 학습 및 평가

### 모델: Random Forest Classifier
* 결정 트리 100개를 만들어 다수결로 예측
* 주요 파라미터: `n_estimators=100`, `max_depth=10`
* 선택 이유: 피처 중요도 확인이 쉽고, 과적합에 비교적 강하며, 해석이 용이함

### 평가 지표
* Classification Report (Precision, Recall, F1-score)
* Confusion Matrix
* Feature Importance (어떤 코드 메트릭이 결함과 관련이 깊은지)

In [ ]:
# ==================== 5. 모델 학습 및 평가 ====================
# 예측 모델 정의 및 학습
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_resampled, y_train_resampled)
print("[AI 예측 모델 학습 완료]")

# 평가용 데이터로 예측 수행
y_pred = rf_model.predict(X_test)

# 모델 성능 평가지표 출력
print("\n=== 분류 예측 성능 지표 (Classification Report) ===")
print(classification_report(y_test, y_pred))

# Confusion Matrix 히트맵
plt.figure(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Normal(0)', 'Predicted Defect(1)'],
            yticklabels=['Actual Normal(0)', 'Actual Defect(1)'])
plt.title('Actual vs Predicted Values (Confusion Matrix)')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# Feature Importance
feature_importances = pd.Series(
    rf_model.feature_importances_, index=X.columns
).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=feature_importances.values, y=feature_importances.index, palette='viridis')
plt.title('Feature Importance (Key Code Metrics Causing Defects)')
plt.xlabel('Importance Score')
plt.ylabel('Code Metrics')
plt.tight_layout()
plt.show()

# 5. 신규 데이터 예측 및 엑셀 리포트 생성

* 실제 사용 시에는 `pd.read_excel('신규_모듈_테스트.xlsx')`로 교체
* 모델이 예측한 결함 여부에 따라 QA Action 가이드를 자동 생성
  - 결함(1) → 결함 위험 (집중 테스트 요망)
  - 정상(0) → 정상 (일반 통과)
* 결과를 엑셀 파일로 저장하여 실무 부서에 전달
  * 리포트 결과: 결함 모듈 (1건, Index 1671): 코드가 길고 복잡도가 높음.

In [ ]:
# ==================== 6. 신규 데이터 예측 및 엑셀 리포트 생성 ====================
# 원본 데이터 전체 추출
tester_dataset = df_concat.drop(columns=['defects', 'project_source', 'id'], errors='ignore').copy()

# AI 예측 수행 (예측 시 학습에 사용하지 않은 8개 중복 지표는 제외)
new_predictions = rf_model.predict(tester_dataset.drop(columns=['e', 'n', 'v', 'b', 'total_Op', 'total_Opnd', 'branchCount', 'lOCode'], errors='ignore'))

# 예측 결과를 데이터프레임에 추가
tester_dataset['Predicted_Defect'] = new_predictions
tester_dataset['QA_Action'] = tester_dataset['Predicted_Defect'].apply(
    lambda x: '결함 위험 (집중 테스트 요망)' if x == 1 else '정상 (일반 통과)'
)

# 최종 리포트 엑셀 저장
tester_dataset.to_excel('AI_결함예측_리포트.xlsx', index=False)
print("엑셀 기반 결함 예측 리포트가 성공적으로 생성되었습니다.")

# 주요 컬럼만 상위 5개 확인 (원본 지표인 'e'도 포함)
display(tester_dataset[['loc', 'v(g)', 'e', 'Predicted_Defect', 'QA_Action']].head())

### AI 모델 성능 평가 지표 요약

| 구분 | 수치 | 의미 |
| :--- | :---: | :--- |
| **Accuracy** | 0.77 | 전체 샘플 중 77%를 올바르게 맞힘 |
| **정상(0) F1-Score** | 0.85 | 정상 모듈 예측에 대한 종합 성능 (높은 성능) |
| **결함(1) Precision** | 0.41 | 결함이라고 예측한 것 중 실제 결함인 비율 (정밀도) |
| **결함(1) Recall** | 0.52 | 실제 결함 전체 중 모델이 찾아낸 비율 (재현율) |
| **결함(1) F1-Score** | 0.46 | 결함 예측에 대한 종합 성능 지표 |







*   전체 Accuracy (정확도)는 77%이며, 정상 모듈은 안정적으로 구분.
*   결함 탐지 Recall은 52%로, 실제 결함의 절반 정도를 사전에 포착가능.
*   소프트웨어 결함 예측 특성상 데이터가 불균형하기 때문에 이 정도 성능은 실무적으로 유의미



In [ ]:
# [코랩 마지막 셀에 추가하여 실행] 모델을 pkl 파일로 저장하고 다운로드하기
import joblib
from google.colab import files

# 학습이 완료된 rf_model 객체를 파일로 저장
joblib.dump(rf_model, 'rf_model.pkl')
print("rf_model.pkl 파일이 생성되었습니다.")

# 내 컴퓨터로 파일 다운로드
files.download('rf_model.pkl')